# Data Quality & Validation

In [2]:
from scipy.stats import chi2_contingency

table = [
    [8, 151],    # Business
    [1, 13],     # Enterprise
    [33, 440],   # Growth
    [44, 510]    # Starter
]

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)
print("Expectd frequency:", expected)

Chi-square statistic: 1.6161202835044577
p-value: 0.6557402024616013
Degrees of freedom: 3
Expectd frequency: [[ 11.395      147.605     ]
 [  1.00333333  12.99666667]
 [ 33.89833333 439.10166667]
 [ 39.70333333 514.29666667]]


Churn vs Industry

In [3]:
from scipy.stats import chi2_contingency

table = [
    [7, 106],    # Education
    [11, 191],   # Financial Services
    [14, 162],   # Healthcare
    [7, 82],     # Logistics
    [10, 141],   # Manufacturing
    [14, 150],   # Retail
    [23, 282]    # Software
]

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)
print("Expected frequency:", expected)

Chi-square statistic: 1.8833885113468554
p-value: 0.9301047054906803
Degrees of freedom: 6
Expected frequency: [[  8.09833333 104.90166667]
 [ 14.47666667 187.52333333]
 [ 12.61333333 163.38666667]
 [  6.37833333  82.62166667]
 [ 10.82166667 140.17833333]
 [ 11.75333333 152.24666667]
 [ 21.85833333 283.14166667]]


Country wise

In [4]:
from scipy.stats import chi2_contingency

table = [
    [14, 91],    # Australia
    [7, 109],    # Brazil
    [5, 132],    # Canada
    [8, 119],    # France
    [12, 112],   # Germany
    [14, 104],   # India
    [5, 119],    # Japan
    [8, 118],    # Netherlands
    [3, 101],    # United Kingdom
    [10, 109]    # United States
]

chi2, p_value, dof, expected = chi2_contingency(table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)
print("Expected frequency:", expected)


Chi-square statistic: 19.102458327769593
p-value: 0.024335672001983135
Degrees of freedom: 9
Expected frequency: [[  7.525       97.475     ]
 [  8.31333333 107.68666667]
 [  9.81833333 127.18166667]
 [  9.10166667 117.89833333]
 [  8.88666667 115.11333333]
 [  8.45666667 109.54333333]
 [  8.88666667 115.11333333]
 [  9.03       116.97      ]
 [  7.45333333  96.54666667]
 [  8.52833333 110.47166667]]


pvalue<0.05 = we reject null hypothesis

Which countries are contributing most to the significant Country → Churn association?


In [5]:
import numpy as np
from scipy.stats import chi2_contingency

countries = [
    "Australia", "Brazil", "Canada", "France", "Germany",
    "India", "Japan", "Netherlands", "United Kingdom", "United States"
]

table = np.array([
    [14, 91],
    [7, 109],
    [5, 132],
    [8, 119],
    [12, 112],
    [14, 104],
    [5, 119],
    [8, 118],
    [3, 101],
    [10, 109]
])

chi2, p_value, dof, expected = chi2_contingency(table)

residuals = (table - expected) / np.sqrt(expected)

for country, residual in zip(countries, residuals[:, 0]):
    print(f"{country}: {residual:.2f}")

Australia: 2.36
Brazil: -0.46
Canada: -1.54
France: -0.37
Germany: 1.04
India: 1.91
Japan: -1.30
Netherlands: -0.34
United Kingdom: -1.63
United States: 0.50


In [4]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

In [27]:
# testing the connection
with engine.connect() as connection:
    print("PostgreSQL connection successful!")

PostgreSQL connection successful!


In [28]:
import pandas as pd

query = """
SELECT
    subscription_status,
    user_count
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

print(df.head())
print(df.dtypes)

  subscription_status  user_count
0              active           3
1              active          11
2              active          35
3              active         108
4              active          20
subscription_status      str
user_count             int64
dtype: object


In [25]:
from scipy.stats import mannwhitneyu

churned = df.loc[
    df["subscription_status"] == "churned",
    "user_count"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "user_count"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45280.5
p-value: 0.39651614228189047


There is not sufficient statistical evidence that user-count distributions differ between churned and non-churned customers in this dataset.

In [33]:
query = """
SELECT
    subscription_status,
    mrr
FROM customer_360
"""
with engine.connect() as connection:
    df = pd.read_sql(query, connection)

print(df.head())
print(df.dtypes)

  subscription_status      mrr
0              active     36.0
1              active    288.0
2              active    840.0
3              active  14852.0
4              active    648.0
subscription_status        str
mrr                    float64
dtype: object


In [34]:
churned = df.loc[
    df["subscription_status"] == "churned",
    "mrr"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "mrr"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45771.0
p-value: 0.49086143023843753


In [35]:
query = """
SELECT
    subscription_status,
    seats
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "seats"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "seats"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45846.0
p-value: 0.5062178937671258


In [36]:
query = """
SELECT
    subscription_status,
    total_invoiced
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "total_invoiced"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "total_invoiced"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45607.5
p-value: 0.4587942961950079


In [37]:
query = """
SELECT
    subscription_status,
    employee_count
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "employee_count"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "employee_count"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 45569.0
p-value: 0.4511845381743994


In [38]:
query = """
SELECT
    subscription_status,
    users_per_seat
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "users_per_seat"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "users_per_seat"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 44551.5
p-value: 0.27791845868908716


In [40]:
query = """
SELECT
    subscription_status,
    COALESCE(ticket_count, 0) AS ticket_count
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "ticket_count"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "ticket_count"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 44293.0
p-value: 0.2390577411850029


In [41]:
query = """
SELECT
    subscription_status,
    COALESCE(high_priority_tickets, 0) AS high_priority_tickets
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "high_priority_tickets"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "high_priority_tickets"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 47736.0
p-value: 0.9545764106800472


In [42]:
query = """
SELECT
    subscription_status,
    COALESCE(urgent_tickets, 0) AS urgent_tickets
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "urgent_tickets"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "urgent_tickets"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 46641.0
p-value: 0.6043824178213881


In [43]:
query = """
SELECT
    subscription_status,
    avg_satisfaction
FROM customer_360
WHERE avg_satisfaction IS NOT NULL
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "avg_satisfaction"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "avg_satisfaction"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 50374.0
p-value: 0.14784654655538257


In [44]:
query = """
SELECT
    subscription_status,
    avg_resolution_hours
FROM customer_360
WHERE avg_resolution_hours IS NOT NULL
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "avg_resolution_hours"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "avg_resolution_hours"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 42206.5
p-value: 0.20382565534870756


In [45]:
query = """
SELECT
    subscription_status,
    COALESCE(unresolved_tickets, 0) AS unresolved_tickets
FROM customer_360
"""

with engine.connect() as connection:
    df = pd.read_sql(query, connection)

churned = df.loc[
    df["subscription_status"] == "churned",
    "unresolved_tickets"
]

not_churned = df.loc[
    df["subscription_status"] != "churned",
    "unresolved_tickets"
]

statistic, p_value = mannwhitneyu(
    churned,
    not_churned,
    alternative="two-sided"
)

print("Mann-Whitney U statistic:", statistic)
print("p-value:", p_value)

Mann-Whitney U statistic: 48495.0
p-value: 0.8162084129100544


In [46]:
import numpy as np
from scipy.stats import chi2_contingency

table = np.array([
    [14, 91],    # Australia
    [7, 109],    # Brazil
    [5, 132],    # Canada
    [8, 119],    # France
    [12, 112],   # Germany
    [14, 104],   # India
    [5, 119],    # Japan
    [8, 118],    # Netherlands
    [3, 101],    # United Kingdom
    [10, 109]    # United States
])

chi2, p_value, dof, expected = chi2_contingency(table)

n = table.sum()
r, k = table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, k - 1))
)

print("Cramer's V:", cramers_v)

Cramer's V: 0.12616939118954854


In [48]:
# ============================================================
# PHASE 7 — FINAL ML EXPERIMENT
# Historical Churn Classification from customer_360
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------------------
# 1. Load customer_360
# ------------------------------------------------------------

df = pd.read_sql(
    "SELECT * FROM customer_360",
    engine
)

print("customer_360:", df.shape)


# ------------------------------------------------------------
# 2. Create target
# ------------------------------------------------------------

df["churned"] = (
    df["subscription_status"] == "churned"
).astype(int)

print("\nTarget distribution:")
print(df["churned"].value_counts())

print("\nChurn rate:")
print(round(df["churned"].mean() * 100, 2), "%")


# ------------------------------------------------------------
# 3. Create useful pre-outcome features
# ------------------------------------------------------------

df["customer_tenure_days"] = (
    pd.to_datetime(df["started_on"])
    - pd.to_datetime(df["signup_date"])
).dt.days


# ------------------------------------------------------------
# 4. Remove explicit leakage / outcome fields
# ------------------------------------------------------------

drop_cols = [
    "account_id",

    # Direct outcome information
    "subscription_status",
    "ended_on",
    "subscription_duration",

    # Raw dates
    "signup_date",
    "started_on",

    # Aggregates that can directly encode the outcome
    # through post-outcome financial activity
    "uncollectible_amount"
]

X = df.drop(
    columns=drop_cols + ["churned"],
    errors="ignore"
)

y = df["churned"]


# ------------------------------------------------------------
# 5. Identify feature types
# ------------------------------------------------------------

categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)


# ------------------------------------------------------------
# 6. Preprocessing
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


# ------------------------------------------------------------
# 7. Stratified train/test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)


# ------------------------------------------------------------
# 8. Model 1 — Logistic Regression
# ------------------------------------------------------------

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_prob = logistic_model.predict_proba(X_test)[:, 1]
logistic_pred = logistic_model.predict(X_test)

print("\n==============================")
print("LOGISTIC REGRESSION")
print("==============================")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, logistic_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        logistic_pred,
        digits=4
    )
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, logistic_prob), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_test, logistic_prob), 4)
)


# ------------------------------------------------------------
# 9. Model 2 — Random Forest
# ------------------------------------------------------------

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=3
    ))
])

rf_model.fit(X_train, y_train)

rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict(X_test)

print("\n==============================")
print("RANDOM FOREST")
print("==============================")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        digits=4
    )
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, rf_prob), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_test, rf_prob), 4)
)


# ------------------------------------------------------------
# 10. Cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_cv = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_cv = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("\n==============================")
print("5-FOLD CROSS-VALIDATION")
print("==============================")

print(
    "Logistic Regression ROC-AUC:",
    round(logistic_cv.mean(), 4),
    "+/-",
    round(logistic_cv.std(), 4)
)

print(
    "Random Forest ROC-AUC:",
    round(rf_cv.mean(), 4),
    "+/-",
    round(rf_cv.std(), 4)
)


# ------------------------------------------------------------
# 11. Feature importance — Random Forest
# ------------------------------------------------------------

fitted_preprocessor = rf_model.named_steps["preprocessor"]
fitted_rf = rf_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": fitted_rf.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print("\n==============================")
print("TOP 15 RANDOM FOREST FEATURES")
print("==============================")

print(
    importance_df.head(15).to_string(index=False)
)


# ------------------------------------------------------------
# 12. Final comparison
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "test_roc_auc": [
        roc_auc_score(y_test, logistic_prob),
        roc_auc_score(y_test, rf_prob)
    ],
    "test_pr_auc": [
        average_precision_score(y_test, logistic_prob),
        average_precision_score(y_test, rf_prob)
    ],
    "cv_roc_auc_mean": [
        logistic_cv.mean(),
        rf_cv.mean()
    ],
    "cv_roc_auc_std": [
        logistic_cv.std(),
        rf_cv.std()
    ]
})

print("\n==============================")
print("FINAL MODEL COMPARISON")
print("==============================")

print(
    comparison.round(4).to_string(index=False)
)

print("\nPhase 7 ML experiment completed.")

customer_360: (1200, 30)

Target distribution:
churned
0    1114
1      86
Name: count, dtype: int64

Churn rate:
7.17 %

Categorical features:
['company_name', 'industry', 'country', 'plan']

Numeric features:
['employee_count', 'user_count', 'admin_count', 'member_count', 'viewer_count', 'seats', 'mrr', 'users_per_seat', 'invoice_count', 'total_invoiced', 'avg_invoice_amount', 'paid_amount', 'open_amount', 'ticket_count', 'high_priority_tickets', 'urgent_tickets', 'avg_satisfaction', 'avg_resolution_hours', 'unresolved_tickets', 'customer_tenure_days']

Train: (960, 24)
Test : (240, 24)

LOGISTIC REGRESSION

Confusion Matrix:
[[190  33]
 [ 15   2]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9268    0.8520    0.8879       223
           1     0.0571    0.1176    0.0769        17

    accuracy                         0.8000       240
   macro avg     0.4920    0.4848    0.4824       240
weighted avg     0.8652    0.8000    0.8304  

In [49]:
# ============================================================
# PHASE 8 — CUSTOMER DECISION LAYER
# Customer Retention Intelligence
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load customer_360
# ------------------------------------------------------------

df = pd.read_sql(
    "SELECT * FROM customer_360",
    engine
)

print("Loaded customer_360:", df.shape)


# ------------------------------------------------------------
# 2. Create business signals
# ------------------------------------------------------------

# Missing support metrics = zero activity
df["ticket_count"] = df["ticket_count"].fillna(0)
df["high_priority_tickets"] = df["high_priority_tickets"].fillna(0)
df["urgent_tickets"] = df["urgent_tickets"].fillna(0)
df["unresolved_tickets"] = df["unresolved_tickets"].fillna(0)

# Financial metrics
df["mrr"] = df["mrr"].fillna(0)
df["open_amount"] = df["open_amount"].fillna(0)
df["uncollectible_amount"] = df["uncollectible_amount"].fillna(0)

# ------------------------------------------------------------
# 3. Rule-based risk signals
# ------------------------------------------------------------

# Low engagement
df["low_engagement"] = (
    df["user_count"] <= 5
).astype(int)

# Low commercial value
df["low_mrr"] = (
    df["mrr"] < df["mrr"].median()
).astype(int)

# Payment risk
df["payment_risk"] = (
    (df["open_amount"] > 0) |
    (df["uncollectible_amount"] > 0)
).astype(int)

# Support pressure
df["support_pressure"] = (
    (df["high_priority_tickets"] > 0) |
    (df["urgent_tickets"] > 0)
).astype(int)

# Unresolved support issues
df["unresolved_risk"] = (
    df["unresolved_tickets"] > 0
).astype(int)

# Low satisfaction
df["low_satisfaction"] = (
    df["avg_satisfaction"].fillna(
        df["avg_satisfaction"].median()
    ) < 3.5
).astype(int)


# ------------------------------------------------------------
# 4. Customer Health Score
# ------------------------------------------------------------

risk_columns = [
    "low_engagement",
    "low_mrr",
    "payment_risk",
    "support_pressure",
    "unresolved_risk",
    "low_satisfaction"
]

df["risk_points"] = df[risk_columns].sum(axis=1)

# 100 = healthiest
df["health_score"] = (
    100 -
    (df["risk_points"] / len(risk_columns) * 100)
).round(0)


# ------------------------------------------------------------
# 5. Health segments
# ------------------------------------------------------------

df["health_segment"] = pd.cut(
    df["health_score"],
    bins=[-1, 39, 59, 79, 100],
    labels=[
        "Critical",
        "At Risk",
        "Watch",
        "Healthy"
    ]
)


# ------------------------------------------------------------
# 6. Revenue exposure
# ------------------------------------------------------------

# Existing churned revenue
df["churned_revenue"] = np.where(
    df["subscription_status"] == "churned",
    df["mrr"],
    0
)

# Active customers currently carrying risk signals
df["active_at_risk"] = (
    (df["subscription_status"] == "active") &
    (df["risk_points"] >= 2)
).astype(int)

df["active_revenue_at_risk"] = np.where(
    df["active_at_risk"] == 1,
    df["mrr"],
    0
)


# ------------------------------------------------------------
# 7. Customer intelligence table
# ------------------------------------------------------------

customer_intelligence = df[
    [
        "account_id",
        "company_name",
        "industry",
        "country",
        "plan",
        "subscription_status",
        "mrr",
        "seats",
        "user_count",
        "users_per_seat",
        "ticket_count",
        "avg_satisfaction",
        "avg_resolution_hours",
        "open_amount",
        "uncollectible_amount",
        "risk_points",
        "health_score",
        "health_segment",
        "active_at_risk",
        "active_revenue_at_risk"
    ]
].copy()


# ------------------------------------------------------------
# 8. Push to PostgreSQL
# ------------------------------------------------------------

customer_intelligence.to_sql(
    "customer_intelligence",
    engine,
    if_exists="replace",
    index=False
)

print("\ncustomer_intelligence created successfully.")


# ------------------------------------------------------------
# 9. Executive summary
# ------------------------------------------------------------

print("\n==============================")
print("EXECUTIVE RETENTION SUMMARY")
print("==============================")

print(
    "\nCustomers:",
    len(customer_intelligence)
)

print(
    "Healthy:",
    (customer_intelligence["health_segment"] == "Healthy").sum()
)

print(
    "Watch:",
    (customer_intelligence["health_segment"] == "Watch").sum()
)

print(
    "At Risk:",
    (customer_intelligence["health_segment"] == "At Risk").sum()
)

print(
    "Critical:",
    (customer_intelligence["health_segment"] == "Critical").sum()
)

print(
    "\nActive customers:",
    (
        customer_intelligence["subscription_status"]
        == "active"
    ).sum()
)

print(
    "Active customers flagged at risk:",
    customer_intelligence["active_at_risk"].sum()
)

print(
    "\nCurrent MRR:",
    round(
        customer_intelligence["mrr"].sum(),
        2
    )
)

print(
    "Active MRR exposed to risk:",
    round(
        customer_intelligence[
            "active_revenue_at_risk"
        ].sum(),
        2
    )
)

print(
    "MRR from historically churned customers:",
    round(
        customer_intelligence[
            "mrr"
        ][
            customer_intelligence[
                "subscription_status"
            ] == "churned"
        ].sum(),
        2
    )
)


# ------------------------------------------------------------
# 10. Health segment summary
# ------------------------------------------------------------

segment_summary = (
    customer_intelligence
    .groupby("health_segment", observed=True)
    .agg(
        customers=("account_id", "count"),
        total_mrr=("mrr", "sum"),
        avg_mrr=("mrr", "mean"),
        avg_health_score=("health_score", "mean")
    )
    .sort_values("avg_health_score")
)

print("\n==============================")
print("HEALTH SEGMENTS")
print("==============================")

print(
    segment_summary.round(2).to_string()
)


# ------------------------------------------------------------
# 11. Highest-value active customers needing attention
# ------------------------------------------------------------

priority_customers = (
    customer_intelligence[
        customer_intelligence["active_at_risk"] == 1
    ]
    .sort_values(
        ["mrr", "risk_points"],
        ascending=[False, False]
    )
    .head(20)
)

print("\n==============================")
print("TOP 20 ACTIVE CUSTOMERS FOR REVIEW")
print("==============================")

print(
    priority_customers[
        [
            "account_id",
            "company_name",
            "plan",
            "mrr",
            "risk_points",
            "health_score",
            "health_segment",
            "open_amount",
            "uncollectible_amount",
            "ticket_count",
            "avg_satisfaction"
        ]
    ].to_string(index=False)
)

print("\nPhase 8 decision layer completed.")

Loaded customer_360: (1200, 30)

customer_intelligence created successfully.

EXECUTIVE RETENTION SUMMARY

Customers: 1200
Healthy: 235
Watch: 338
At Risk: 309
Critical: 318

Active customers: 1030
Active customers flagged at risk: 822

Current MRR: 1020471.0
Active MRR exposed to risk: 607978.0
MRR from historically churned customers: 58530.0

HEALTH SEGMENTS
                customers  total_mrr  avg_mrr  avg_health_score
health_segment                                                 
Critical              318    35304.0   111.02             27.42
At Risk               309   187828.0   607.86             50.00
Watch                 338   459242.0  1358.70             67.00
Healthy               235   338097.0  1438.71             85.31

TOP 20 ACTIVE CUSTOMERS FOR REVIEW
 account_id              company_name       plan     mrr  risk_points  health_score health_segment  open_amount  uncollectible_amount  ticket_count  avg_satisfaction
     457669      Unified Ops Partners Enterprise 25